# Description:

This notebook provides a collection of analysis functions for multicomponent glass structures based on LAMMPS dump files.
It supports structural characterization using atomic-level information based on the features described below.

## Available Analyses:

- Coordination Numbers: Computes how many neighbors each atom has within a cutoff.
- Bridging and Non-Bridging Oxygens: Determines network connectivity through oxygen coordination.
- Qn Distribution: Counts the number of bridging oxygens attached to each former (e.g., Si), yielding Qn statistics.
- Network Connectivity: Computes average Qⁿ value across the system.
- Bond Angle Distribution: Calculates the angles between triplets of atoms to assess structural order.
- Radial distribution function: calculate the pairwise radial distribution functions g_ij(r) for all pairs of types i-j, normalized to ideal gas. it also calculate the cumulative coordination number CN_ij(r) obtained by integrating the RDF up to each r. 
- Guttmann Rings: get the rings size distribution following Guttmann algorithm.


## Input Requirements:
Currently, all functions assume LAMMPS dump file format (optionally gzipped). The simulation cell needs to be cubic or orthorhombic.
The required fields in the dump file are: atom ID, type, coordinates (wrapped or unwrapped), and box dimensions.

## Main Functions:

- `read_lammps_dump()`: Read atom data and box size.
- `remove_atom_type()`: Remove unwanted atom types.
- `compute_cell_list()`, `get_neighbors()`: neighbor search using a cell list.
- `compute_coordination()`: Coordination histogram for a given atom type.
- `compute_Qn()`: Compute Qⁿ and partial Qⁿ distributions.
- `compute_network_connectivity()`: Calculate the average Qⁿ.
- `compute_angles()`: Compute bond angle distributions (partial implementation).
- `compute_rdf()`: Compute radial distributions function and cumulative coordination number of all pairs in the sample.
- `write_distribution_to_file()`: Output histogram data to file.
- `compute_guttmann_rings()`: get guttmann rings statistics.



In [ ]:
import matplotlib.pyplot as plt
import ase
from ase.io import read

import pyiron_glass 
from pyiron_glass.io_utils import get_properties_for_structure_analysis, write_distribution_to_file
from pyiron_glass.analysis.qn_network_connectivity import compute_qn, compute_network_connectivity
from pyiron_glass.analysis.radial_distribution_functions import compute_coordination, compute_rdf
from pyiron_glass.analysis.bond_angle_distribution import compute_angles


# Coordination numbers and $Q^n$:

The cell below serves as a show case for calculating:

- Coordination Numbers: Computes how many neighbors each atom has within a cutoff.
- Bridging and Non-Bridging Oxygens: Determines network connectivity through oxygen coordination.
- Qn Distribution: Counts the number of bridging oxygens attached to each former (e.g., Si), yielding Qn statistics.
- Network Connectivity: Computes average Qⁿ value across the system.


In [ ]:
filename = "data/SiONa_25.dump.gz"
cutoff_map = {  # Cutoff distances for different atom types
    "O": 2.0,
    "Si": 2.0,
    "Na": 3.0,
}
# Atom type mapping (as per LAMMPS input)
type_map = {
    1: "O",
    2: "Si",
    3: "Na",
}

network_formers = {"Si"}  # , 'B', 'Al'}
modifiers = {"Na"}  # , 'Ca'}
O_type = [t for t, e in type_map.items() if e == "O"]
former_types = [t for t, e in type_map.items() if e in network_formers]
modifier_types = [t for t, e in type_map.items() if e in modifiers]

atoms = read(filename, format="lammps-dump-text")

ids, types, coords, box_size = get_properties_for_structure_analysis(atoms)

print("IDs:", len(ids), ids)
print("Types:", len(types), types)
print("Coordinates:", len(coords), coords)
print("Box Size:", box_size)

# Coordination distributions
O_coord, _ = compute_coordination(
    ids, types, coords, box_size, O_type, cutoff_map["O"], former_types
)

former_coords = {
    type_map[t]: compute_coordination(
        ids, types, coords, box_size, [t], cutoff_map[type_map[t]], O_type
    )[0]
    for t in former_types
}

modifier_coords = {
    type_map[t]: compute_coordination(
        ids, types, coords, box_size, [t], cutoff_map[type_map[t]], O_type
    )[0]
    for t in modifier_types
}

Qn_dist, _ = compute_qn(
    ids, types, coords, box_size, cutoff_map["O"], former_types, O_type
)

# Print results
print("\nO_n distribution:")

for n, c in O_coord.items():
    print(f"O_{n}: {c}")

for former, coord in former_coords.items():
    print(f"\n{former}_n distribution:")
    for n, c in coord.items():
        print(f"{former}_{n}: {c}")

for mod, coord in modifier_coords.items():
    print(f"\n{mod}_n distribution:")
    for n, c in coord.items():
        print(f"{mod}_{n}: {c}")

print("\nQn distribution:")
for n, c in sorted(Qn_dist.items()):
    print(f"Q_{n}: {c}")

total_formers = sum(len(v) for v in former_coords.values())
net_conn = compute_network_connectivity(Qn_dist)
print(f"\nNetwork connectivity: {net_conn:.4f}")


x = 45
# Write O_n distribution
write_distribution_to_file(x, "O_n.dat", O_coord, "O")

# Write each former type distribution
for former, coord in former_coords.items():
    write_distribution_to_file(x, f"{former}_n.dat", coord, former)

# Write each modifier type distribution
for mod, coord in modifier_coords.items():
    write_distribution_to_file(x, f"{mod}_n.dat", coord, mod)

# Write Qn distribution
write_distribution_to_file(x, "Qn.dat", Qn_dist, "Q")


# Bond angle distribution:

The cell below serves as a show case for calculating:

- Bond angle distribution

In [ ]:
bond_angle_distribution = compute_angles(
    types,
    coords,
    box_size,
    center_type=[2],
    neighbor_type=[1],  # Assuming Si is the center for O-Si-O angles
    cutoff=2,
    bins=180,
)

# Unpack the result to ensure correct usage
angles, counts = bond_angle_distribution

plt.plot(
    angles,
    counts,
    label="O-Si-O angles",
    color="blue",
)
plt.xlabel("Angle (degrees)")
plt.ylabel("Frequency")
plt.legend()
plt.show()

# Bond angle distribution:

The cell below serves as a show case for calculating:

- radial distribution function

In [ ]:
r, rdfs, cumcn = compute_rdf(coords=coords, 
                             types=types, 
                             box_size=box_size, 
                             r_max=10, 
                             n_bins=1000)



fig, ax = plt.subplots(1, 1, figsize=(3, 3), dpi=200, layout='constrained')

ax.plot(r, rdfs[(1, 1)], "-", color="C0", label=r'$g_{1-1}(r)$')
ax.plot(r, rdfs[(2, 1)], "-", color="C1", label=r'$g_{2-1}(r)$')
ax.plot(r, rdfs[(3, 1)], "-", color="C2", label=r'$g_{3-1}(r)$')
ax.plot(r, cumcn[(1, 1)], "--", color="C0", label=r'$CN_{1-1}(r)$')
ax.plot(r, cumcn[(2, 1)], "--", color="C1", label=r'$CN_{2-1}(r)$')
ax.plot(r, cumcn[(3, 1)], "--", color="C2", label=r'$CN_{3-1}(r)$')
ax.set_xlim(0, 5)
ax.set_ylim(0, 25)
ax.set_xlabel(r"$r (\AA)$")
ax.set_ylabel(r"$g(r), CN(r)$")
#ax.legend()

plt.show()


# Rings Calculation:

The cell below serves as a show case for calculating:

- Guttman rings statistics

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ase.io import read

from pyiron_glass.io_utils import get_properties_for_structure_analysis
from pyiron_glass.analysis.rings import compute_guttmann_rings, generate_bond_length_dict

filename = "data/SiONa_25.dump.gz"
atoms = read(filename, format="lammps-dump-text")

mapping = {1: 'O', 2: 'Si', 3: 'Na'}
symbols = [mapping[int(t)] for t in atoms.get_atomic_numbers()]  # or atoms.numbers
atoms.symbols = symbols  # updates atomic numbers as well


ids, types, coords, box_size = get_properties_for_structure_analysis(atoms)


specific_cutoffs = {
    ("Si", "O"): 2.0,
}

bond_lengths = generate_bond_length_dict(atoms, specific_cutoffs=specific_cutoffs)

rings_dist, mean_ring_size = compute_guttmann_rings(types=types, 
                                                    coords=coords, 
                                                    box_size=box_size,
                                                    bond_lengths=bond_lengths, 
                                                    max_size=40, 
                                                    n_cpus=1)



sizes = np.array(sorted(rings_dist.keys()))
counts = np.array([rings_dist[s] for s in sizes])


fig, ax = plt.subplots(1, 1, figsize=(3.5, 3.5/1.333333), dpi=200, layout='constrained')

ax.bar(sizes / 2, counts / counts.sum(), width=0.4, align="center", linewidth=1, edgecolor="black", facecolor="C1")
ax.set_xlim(2, (sizes/2).max()+1)
ax.set_xticks(np.arange(2, (sizes/2).max()+1, 1))
ax.set_ylim(0, 0.5)
ax.set_xlabel(r"Si atoms in ring ($\#$)")
ax.set_ylabel(r"Normalized count")

plt.show()
